In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("./abstract_metadata_with_ideal_spans_correct_no_spans_with_min4chars_df_en_cz_final_with_selected_8_docs.tsv", sep = '\t')

In [3]:
first_4_text_list = (3, 7, 6, 19)

In [4]:
filtered_df_first_4 = df[df['hal_id'].isin(first_4_text_list)]


In [5]:
filtered_df_first_4


,hal_id,translation_id,sentence_indices,num_sentences,total_characters,source,total_spans,total_span_chars,span_char_ratio_pct
0,3,3,"[0, 1, 2, 3, 4, 5, 6, 7]",8,1785,Using data and artificial intelligence to try ...,42,600,33.61
1,7,7,"[24, 25, 26, 27, 28, 29, 30, 31]",8,1625,LNG investments hit record in 2019 & the bigge...,29,401,24.68
6,6,6,"[16, 17, 18, 19, 20, 21, 22, 23]",8,1367,Three Scottish students named among Europe's b...,30,574,41.99
7,19,19,"[56, 57, 58, 59, 60, 61, 62, 63]",8,1356,"US sends troops, air defense systems to Saudi ...",35,457,33.70


In [6]:
values = filtered_df_first_4[['hal_id','total_spans']]
values

,hal_id,total_spans
0,3,42
1,7,29
6,6,30
7,19,35


In [7]:
import json

In [8]:
path_json = "spans_ideal_df_en_cz_final_with_selected_8_docs.json"
# Load JSON data
with open(path_json, 'r', encoding='utf-8') as f:
    json_data_ideal_spans = json.load(f)

In [9]:
print(len(json_data_ideal_spans))

64


**Check which indices/sentences do not have any spans**

In [10]:
for index, data in enumerate(json_data_ideal_spans):
    if not data:
        print(index)

16


In [11]:
import pandas as pd
path_csv = "postedition_aligned_with_tokenized_offsets_df_en_cz_final_with_selected_8_docs.csv"
df_en_cz = pd.read_csv(path_csv)

**Get sentence length for each abstract**

In [12]:
def sentence_length_abstract(translations):
    sentences_length_of_abstract = []
    for idx, value in translations.items():
        sentences_length_of_abstract.append(len(value))
    return sentences_length_of_abstract


In [13]:
first_4_text_list

(3, 7, 6, 19)

In [14]:
import ast
sentences_length_of_abstract = []
for abstract_id in first_4_text_list:
    row = filtered_df_first_4[filtered_df_first_4['translation_id'] == abstract_id]
    indices = ast.literal_eval(row["sentence_indices"].iloc[0])
    first_index = indices[0]
    last_index = indices[-1]
    print(abstract_id, first_index, last_index)
    
    sentences_length_of_each_abstract = sentence_length_abstract(df_en_cz.iloc[first_index:last_index+1]['translation'])
    sentences_length_of_abstract.append({
        "translation_id":abstract_id,
        "translation_starting_index": first_index,
        "translation_ending_index": last_index,
        "document_length":sentences_length_of_each_abstract
    })

3 0 7
7 24 31
6 16 23
19 56 63


In [15]:
sentences_length_of_abstract[2]

{'translation_id': 6,
 'translation_starting_index': 16,
 'translation_ending_index': 23,
 'document_length': [57, 100, 207, 248, 137, 236, 211, 171]}

In [16]:
print(sum(sentences_length_of_abstract[1]["document_length"]))

1625


In [17]:
spans_all_abstracts = []
for abstract_info in sentences_length_of_abstract:
    translation_starting_index = abstract_info["translation_starting_index"]
    translation_ending_index = abstract_info["translation_ending_index"]
    spans_each_abstract = json_data_ideal_spans[translation_starting_index:translation_ending_index+1]
    spans_all_abstracts.append({
        "translation_id": abstract_info["translation_id"],
        "ideal_spans":spans_each_abstract
    })


In [18]:
spans_all_abstracts[0]

{'translation_id': 3,
 'ideal_spans': [[{'text': ['k', 'vyzkoušení', 'a'], 'start': 31, 'end': 46},
   {'text': ['HSBC'], 'start': 87, 'end': 92},
   {'text': ['vytlačení'], 'start': 95, 'end': 105},
   {'text': ['z', 'jeho'], 'start': 110, 'end': 117},
   {'text': ['klíčová', 'priorita'], 'start': 165, 'end': 182}],
  [{'text': ['jedno'], 'start': 6, 'end': 12},
   {'text': ['je', 'ze'], 'start': 120, 'end': 126},
   {'text': ['podstaty'], 'start': 130, 'end': 139},
   {'text': ['globální'], 'start': 197, 'end': 206},
   {'text': ['klientského'], 'start': 214, 'end': 226},
   {'text': ['společnosti'], 'start': 239, 'end': 251}],
  [{'text': ['uvedla,'], 'start': 66, 'end': 74},
   {'text': ['iniciativa', '„síťový', 'příjem“'], 'start': 83, 'end': 110},
   {'text': ['přinesla'], 'start': 114, 'end': 123},
   {'text': ['stovky'], 'start': 136, 'end': 143}],
  [{'text': ['Pohon'], 'start': 0, 'end': 5},
   {'text': ['bránit', 'svou'], 'start': 39, 'end': 51},
   {'text': ['přítomnost'], 

In [19]:
for index, spans_each_abstract in enumerate(spans_all_abstracts):
    print(len(spans_all_abstracts[index]["ideal_spans"])) #check if all the sentences has ideal spans

8
8
8
8


**Recompute the span offsets for each abstract**

The ideal spans were computed on sentence level. That is why they contain the start and end for each span w.r.t to the sentence length. However, when we show the abstract on the system, we show all the sentences as document. So we need to readjust the span offsets. For en_fr, there was a title and the next sentence started as a new line which is 2 space in python space. But for en_cz, there is no title, so we can add 1 space after the first sentence. 

**Now how to tackle when a sentence does not have any spans**

its handled in line 4 and 23, if the list does not contain any span, then the inner loop will not run and in line line 21, its adding the previous sentence length so that even the loop is not running, the calculation is not hampered.

In [20]:
def compute_absolute_spans(sentence_spans, sentences_length_of_abstract, has_title=False, title_gap=2, sep_gap=1):
    """
    sentence_spans: list of lists — one list of span-dicts per sentence
                     (each span-dict has 'text', 'start', 'end' — offsets local to that sentence)
    sentences_length_of_abstract: list of ints — character length of each sentence, in the SAME order
                       as sentence_spans (must be same length, or one longer if has_title=True
                       and the title's length is sentence_lengths[0])
    has_title: if True, sentence_spans[0] is treated as a title block separated by title_gap
    title_gap: characters between title and first sentence (only used if has_title)
    sep_gap: characters between consecutive sentences (e.g. 1 for a single space)
    """
    spans_with_new_starts_ends = []
    spans_with_new_starts_ends = []
    previous_sentence_length = title_gap if has_title else 0
    for index, spans in enumerate(sentence_spans):
        if index == 0:
            spans_title = []
            for span in spans: #
                if span['start'] == 0:
                    start = 0
                else:
                    start = span['start'] + 1
                spans_title.append({
                                    'text': span['text'][:],
                                    'startnew': start,
                                    'endnew': span['end']

                                    })

            spans_with_new_starts_ends.append(spans_title)


        elif index == 1 and has_title:
            spans_each_sentence = []
            previous_sentence_length += sentences_length_of_abstract[index - 1]

            for ind,span in enumerate(spans):
                print("index in index 1: ", ind)
                print("spans in index 1: ", span)
                if span['start'] == 0:
                    start = span['start'] + previous_sentence_length
                else:
                    start = span['start'] + previous_sentence_length + 1

                spans_each_sentence.append({
                                    'text': span['text'][:],
                                    'startnew': start,
                                    'endnew': span['end'] + previous_sentence_length

                                    })
            spans_with_new_starts_ends.append(spans_each_sentence)

        else:
            spans_each_sentence = []
            previous_sentence_length += sentences_length_of_abstract[index - 1] + sep_gap
            for ind,span in enumerate(spans):
                if span['start'] == 0:
                    start = span['start'] + previous_sentence_length
                else:
                    start = span['start'] + previous_sentence_length + sep_gap
                spans_each_sentence.append({
                                    'text': span['text'][:],
                                    'startnew': start,
                                    'endnew': span['end'] + previous_sentence_length

                                    })

            spans_with_new_starts_ends.append(spans_each_sentence)
    return spans_with_new_starts_ends

 **Match the list length of spans_all_spans and sentences_length_of_abstract**
 
 From the whole dataset, we need to select only the translations which we want to use for exp1 or for ideal case. Here we can get the sentence lengths for each abstract when the **translation_id** matches in both list.
 
     

In [21]:
updated_spans_offsets_all_documents = []
for spans_each_abstract in spans_all_abstracts:
    translation_id = spans_each_abstract['translation_id']
    sentence_spans = spans_each_abstract['ideal_spans']

    # find the single matching entry, then pull its document_length
    matching_entry = next(entry for entry in sentences_length_of_abstract if entry['translation_id'] == translation_id)
    sentences_length = matching_entry['document_length']
    updated_spans_offsets_all_documents.append({
        'translation_id':translation_id,
        'updated_spans_offsets': compute_absolute_spans(sentence_spans, sentences_length)
    })

In [22]:
updated_spans_offsets_all_documents

[{'translation_id': 3,
  'updated_spans_offsets': [[{'text': ['k', 'vyzkoušení', 'a'],
     'startnew': 32,
     'endnew': 46},
    {'text': ['HSBC'], 'startnew': 88, 'endnew': 92},
    {'text': ['vytlačení'], 'startnew': 96, 'endnew': 105},
    {'text': ['z', 'jeho'], 'startnew': 111, 'endnew': 117},
    {'text': ['klíčová', 'priorita'], 'startnew': 166, 'endnew': 182}],
   [{'text': ['jedno'], 'startnew': 235, 'endnew': 240},
    {'text': ['je', 'ze'], 'startnew': 349, 'endnew': 354},
    {'text': ['podstaty'], 'startnew': 359, 'endnew': 367},
    {'text': ['globální'], 'startnew': 426, 'endnew': 434},
    {'text': ['klientského'], 'startnew': 443, 'endnew': 454},
    {'text': ['společnosti'], 'startnew': 468, 'endnew': 479}],
   [{'text': ['uvedla,'], 'startnew': 553, 'endnew': 560},
    {'text': ['iniciativa', '„síťový', 'příjem“'],
     'startnew': 570,
     'endnew': 596},
    {'text': ['přinesla'], 'startnew': 601, 'endnew': 609},
    {'text': ['stovky'], 'startnew': 623, 'endne

**update all the word offsets in the abstracts/documents**

Before we have updated the span offsets w.r.t the merged document. This time we need to do it for all the words as we need to align

here we store the word offsets from **postedition_aligned_with_tokenized_offsets_df_en_cz_final_with_selected_8_docs.csv** for the translation_id we will work for exp1.

each entry in the list will be a dictionary in this format:

`{
'translation_id': translation_id,
'word_offsets': word_offsets
}`

In [23]:
first_4_text_list

(3, 7, 6, 19)

In [24]:
df_en_cz.iloc[0]

id_hal                                                                     3
Translation_id                                                             3
line_id                                                                    1
source                     Using data and artificial intelligence to try ...
translation                Využití dat a umělé inteligence k vyzkoušení a...
postedition                Využití dat a umělé inteligence ke zvýšení výn...
translation_tokenized      ['Využití', 'dat', 'a', 'umělé', 'inteligence'...
translation_word_offset    [(0, 7), (7, 11), (11, 13), (13, 19), (19, 31)...
postedition_tokenized      ['Využití', 'dat', 'a', 'umělé', 'inteligence'...
postedition_word_offset    [(0, 7), (7, 11), (11, 13), (13, 19), (19, 31)...
Name: 0, dtype: object

In [25]:
sentences_length_of_abstract[0]

{'translation_id': 3,
 'translation_starting_index': 0,
 'translation_ending_index': 7,
 'document_length': [227, 257, 159, 236, 246, 182, 245, 233]}

In [26]:
import ast
word_offsets_all_abstract = []
for translation_id in first_4_text_list:
    word_offsets = []
    for value in sentences_length_of_abstract:
        if value['translation_id'] == translation_id:
            start_index = value['translation_starting_index']
            end_index = value['translation_ending_index']
            #Convert the string representations to actual Python objects (list of tuples)
            word_offsets = df_en_cz.iloc[start_index:end_index+1]["translation_word_offset"].apply(ast.literal_eval)

    word_offsets_all_abstract.append({
        'translation_id': translation_id,
        'word_offsets': word_offsets
    })      



In [27]:
word_offsets_all_abstract[0] #List[dict[tuple]]

{'translation_id': 3,
 'word_offsets': 0    [(0, 7), (7, 11), (11, 13), (13, 19), (19, 31)...
 1    [(0, 3), (3, 6), (6, 12), (12, 14), (14, 22), ...
 2    [(0, 4), (4, 7), (7, 15), (15, 24), (24, 26), ...
 3    [(0, 5), (5, 8), (8, 18), (18, 27), (27, 33), ...
 4    [(0, 4), (4, 9), (9, 16), (16, 27), (27, 34), ...
 5    [(0, 6), (6, 14), (14, 19), (19, 22), (22, 29)...
 6    [(0, 5), (5, 12), (12, 16), (16, 24), (24, 29)...
 7    [(0, 10), (10, 18), (18, 30), (30, 35), (35, 4...
 Name: translation_word_offset, dtype: object}

**Now update all the word offsets for each document**

The word offsets we retrieved in the last cell, they are for each sentence. Just like we updated the span offsets w.r.t. the abstracts/documents, we will do that here again.

- [x] can we call the same function here as well? we can but there we are also getting the text for each span which we do not need here, so we will use another function

In [28]:
def update_word_offsets(word_offsets, sentences_length_of_abstract, has_title=False, title_gap=2, sep_gap=1):
    updated_word_offsets = []
    counter = 0
    previous_sentence_length = title_gap if has_title else 0

    for index, value in word_offsets.items():
        if counter == 0:
            updated_offsets_each_sentence = []
            for offset in value:
                if offset[0] == 0:
                    start = 0
                else:
                    start = offset[0] + 1
                updated_offsets_each_sentence.append((start, offset[1]))

            updated_word_offsets.append(updated_offsets_each_sentence)

        elif counter == 1:
            updated_offsets_each_sentence = []
            previous_sentence_length += sentences_length_of_abstract[counter - 1] + sep_gap
            for ind,offset in enumerate(value):
                if ind == 0:
                    start = offset[0]+previous_sentence_length
                else:
                    start = offset[0]+previous_sentence_length + 1
                updated_offsets_each_sentence.append((start, offset[1]+previous_sentence_length))

            updated_word_offsets.append(updated_offsets_each_sentence)
        else:
            updated_offsets_each_sentence = []
            previous_sentence_length += sentences_length_of_abstract[counter - 1] + sep_gap
            for ind,offset in enumerate(value):
                if ind == 0:
                    start = offset[0]+previous_sentence_length
                else:
                    start = offset[0]+previous_sentence_length + 1
                updated_offsets_each_sentence.append((start, offset[1]+previous_sentence_length))
            updated_word_offsets.append(updated_offsets_each_sentence)
        counter += 1
    return updated_word_offsets


In [29]:
# reuse the function we used for updating the span offsets
#lets check what is the type of sentence spans
updated_word_offsets_all_documents = []
for index, word_offsets in enumerate(word_offsets_all_abstract):
    translation_id = word_offsets_all_abstract[index]['translation_id']
    word_offsets = word_offsets_all_abstract[index]['word_offsets']
    
    # find the single matching entry, then pull its document_length
    matching_entry = next(entry for entry in sentences_length_of_abstract if entry['translation_id'] == translation_id)
    sentences_length = matching_entry['document_length']
    updated_word_offsets_all_documents.append({
        'translation_id':translation_id,
        'updated_word_offsets': update_word_offsets(word_offsets, sentences_length)
    })
    

In [30]:
updated_word_offsets_all_documents

[{'translation_id': 3,
  'updated_word_offsets': [[(0, 7),
    (8, 11),
    (12, 13),
    (14, 19),
    (20, 31),
    (32, 33),
    (34, 44),
    (45, 46),
    (47, 54),
    (55, 61),
    (62, 64),
    (65, 73),
    (74, 81),
    (82, 87),
    (88, 92),
    (93, 95),
    (96, 105),
    (106, 110),
    (111, 112),
    (113, 117),
    (118, 126),
    (127, 134),
    (135, 139),
    (140, 141),
    (142, 153),
    (154, 158),
    (159, 162),
    (163, 165),
    (166, 173),
    (174, 182),
    (183, 192),
    (193, 204),
    (205, 213),
    (214, 219),
    (220, 227)],
   [(228, 231),
    (232, 234),
    (235, 240),
    (241, 242),
    (243, 250),
    (251, 261),
    (262, 270),
    (271, 273),
    (274, 282),
    (283, 291),
    (292, 303),
    (304, 305),
    (306, 316),
    (317, 322),
    (323, 328),
    (329, 337),
    (338, 348),
    (349, 351),
    (352, 354),
    (355, 358),
    (359, 367),
    (368, 373),
    (374, 380),
    (381, 382),
    (383, 392),
    (393, 404),
    (405, 40

**We can retrieve sentence start and end from the word offset boundaries**

we will need this for sentence alignment between translation and source table

In [31]:
def get_sentence_length_of_abstract_indexes(updated_word_offsets):

    sentences_length_of_abstract_indexes = []
    for index, sentence in enumerate(updated_word_offsets):

        sentences_length_of_abstract_indexes.append((sentence[0][0], sentence[-1][1]))
    return sentences_length_of_abstract_indexes

In [32]:
sentences_length_of_all_abstract_indexes = []
for word_offsets in updated_word_offsets_all_documents:
    offsets = word_offsets['updated_word_offsets']
    translation_id = word_offsets['translation_id']
    sentences_length_of_all_abstract_indexes.append({
        'translation_id':translation_id,
        'updated_sentence_indexes': get_sentence_length_of_abstract_indexes(offsets)
    })

In [33]:
sentences_length_of_all_abstract_indexes[0]

{'translation_id': 3,
 'updated_sentence_indexes': [(0, 227),
  (228, 485),
  (486, 645),
  (646, 882),
  (883, 1129),
  (1130, 1312),
  (1313, 1558),
  (1559, 1792)]}

**convert the updated spans offsets into list of tuples**

In [34]:
updated_spans_offsets_all_documents[0]

{'translation_id': 3,
 'updated_spans_offsets': [[{'text': ['k', 'vyzkoušení', 'a'],
    'startnew': 32,
    'endnew': 46},
   {'text': ['HSBC'], 'startnew': 88, 'endnew': 92},
   {'text': ['vytlačení'], 'startnew': 96, 'endnew': 105},
   {'text': ['z', 'jeho'], 'startnew': 111, 'endnew': 117},
   {'text': ['klíčová', 'priorita'], 'startnew': 166, 'endnew': 182}],
  [{'text': ['jedno'], 'startnew': 235, 'endnew': 240},
   {'text': ['je', 'ze'], 'startnew': 349, 'endnew': 354},
   {'text': ['podstaty'], 'startnew': 359, 'endnew': 367},
   {'text': ['globální'], 'startnew': 426, 'endnew': 434},
   {'text': ['klientského'], 'startnew': 443, 'endnew': 454},
   {'text': ['společnosti'], 'startnew': 468, 'endnew': 479}],
  [{'text': ['uvedla,'], 'startnew': 553, 'endnew': 560},
   {'text': ['iniciativa', '„síťový', 'příjem“'],
    'startnew': 570,
    'endnew': 596},
   {'text': ['přinesla'], 'startnew': 601, 'endnew': 609},
   {'text': ['stovky'], 'startnew': 623, 'endnew': 629}],
  [{'text

In [35]:
def updated_spans_offsets_to_list_of_tupes(spans_per_doc):
    spans_list_of_tuples = []
    for spans in spans_per_doc:
        for span in spans:
            spans_list_of_tuples.append((span['startnew'], span['endnew']))
    return spans_list_of_tuples

In [36]:
#run the above function
updated_spans_list_of_tuples_all_docs = []
for index, updated_spans in enumerate(updated_spans_offsets_all_documents):
    translation_id = updated_spans['translation_id']
    spans = updated_spans['updated_spans_offsets']
    updated_spans_list_of_tuples_all_docs.append({
        'translation_id': translation_id,
        'updated_spans_list_of_tuples': updated_spans_offsets_to_list_of_tupes(spans)
    })
    

In [37]:
updated_spans_list_of_tuples_all_docs[0]

{'translation_id': 3,
 'updated_spans_list_of_tuples': [(32, 46),
  (88, 92),
  (96, 105),
  (111, 117),
  (166, 182),
  (235, 240),
  (349, 354),
  (359, 367),
  (426, 434),
  (443, 454),
  (468, 479),
  (553, 560),
  (570, 596),
  (601, 609),
  (623, 629),
  (646, 651),
  (686, 697),
  (707, 717),
  (726, 729),
  (834, 837),
  (847, 862),
  (933, 935),
  (976, 984),
  (989, 997),
  (1033, 1044),
  (1120, 1129),
  (1171, 1195),
  (1204, 1212),
  (1243, 1252),
  (1267, 1272),
  (1283, 1290),
  (1330, 1337),
  (1351, 1406),
  (1427, 1445),
  (1449, 1457),
  (1468, 1488),
  (1528, 1536),
  (1545, 1552),
  (1559, 1569),
  (1578, 1589),
  (1595, 1602),
  (1606, 1621),
  (1625, 1635),
  (1665, 1674),
  (1691, 1792)]}

### Now show the final highlightig for four abstracts

get the text and merge them to form the abstracts

In [50]:
first_4_text_list

(3, 7, 6, 19)

In [51]:
all_abstracts_translations_text = []

for translation_id, group in df_en_cz.groupby('Translation_id'):
    group_sorted = group.sort_values('line_id')
    merged_text = ' '.join(group_sorted['translation'].tolist())
    
    all_abstracts_translations_text.append({
        'translation_id': translation_id,
        'translation_text': merged_text
    })
ids_to_keep = (3, 7, 6, 19)

filtered_translations_text = [
    entry for entry in all_abstracts_translations_text
    if entry['translation_id'] in ids_to_keep
]

In [55]:
filtered_translations_text[0]

{'translation_id': 3,
 'translation_text': 'Využití dat a umělé inteligence k vyzkoušení a zvýšení výnosů je součástí širšího tlaku HSBC na vytlačení více z jeho rozsáhlé fyzické sítě a klientských dat, což je klíčová priorita dočasného generálního ředitele Noela Quinna. „Je to jedno z prvních komerčních investic do prevence finanční kriminality a podnikání, které tímto způsobem získáváme, je ze své podstaty nižší riziko a rychlejší vítězství,“ řekl Stuart Nivison, globální vedoucí klientského bankovnictví společnosti HSBC. HSBC se odmítla vyjádřit k tomu, co očekává od nového systému, ale uvedla, že širší iniciativa „síťový příjem“ již přinesla další příjmy stovky milionů dolarů. Pohon je důležitou součástí úsilí banky bránit svou globální přítomnost v době, kdy někteří analytici a investoři říkají, že by měla zmenšovat nebo opouštět trhy, jako jsou Spojené státy, kde dosahuje návratnosti pod náklady na kapitál. HSBC byla nucena investovat stovky milionů dolarů do souladu s finanční k

In [56]:
len(filtered_translations_text[0]['translation_text'])

1792

In [63]:
def create_html_highlights(translation_id, text, offsets):
    
    text_highlight = f"highlight_translation_id_{translation_id}"
    # 1. Store annotations separately (preserves offsets)
    annotations = [{"start": s, "end": e, "text": text[s:e]} for s, e in offsets]

    # 2. Function to generate highlighted HTML
    def generate_html_highlight(text, annotations, color="yellow"):
        """
        Creates HTML with highlighted spans without modifying original text
        Preserves original offsets for future use
        """
        # Sort annotations by start position
        sorted_ann = sorted(annotations, key=lambda x: x["start"])

        html_parts = []
        last_end = 0

        for ann in sorted_ann:
            # Add text before highlight
            html_parts.append(text[last_end:ann["start"]])

            # Add highlighted text
            html_parts.append(f'<span style="background-color: {color};">{ann["text"]}</span>')

            last_end = ann["end"]

        # Add remaining text
        html_parts.append(text[last_end:])

        return "".join(html_parts)

    # 3. Generate and save HTML
    html_output = generate_html_highlight(text, annotations, "green")
    
    with open(f"{text_highlight}.html", "w", encoding="utf-8") as f:
        f.write(f"""<!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>Highlighted Text</title>
        <style>
            body {{ font-family: Arial, sans-serif; line-height: 1.6; }}
            span {{ border-radius: 3px; padding: 0 2px; }}
        </style>
    </head>
    <body>
        <pre>{html_output}</pre>
    </body>
    </html>""")

    print(f"Highlighted HTML saved to {text_highlight}.html")
    return html_output

### what is needed to fix
- [ ]1. in the final list of tuples span, there are spans which are less than 4 characters
    1. should i trim it in the final tuple
    
- [ ] 2. there are very large spans - should we omit those.
- [ ] 3. how the randomly span selection (no spans, 4,8,12) happens?


### highlighting with ideal spans:


In [69]:
html_outputs_all_docs = []
for each_text in filtered_translations_text:
    translation_id = each_text["translation_id"]
    text = each_text['translation_text']
    match = next(
    (row for row in updated_spans_list_of_tuples_all_docs if row['translation_id'] == translation_id),
    None
    )
    span_tuples = match['updated_spans_list_of_tuples'] if match else None
    html_outputs_all_docs.append({
        'translation_id': translation_id,
        'html_output': create_html_highlights(translation_id, text, span_tuples)
    })

Highlighted HTML saved to highlight_translation_id_3.html
Highlighted HTML saved to highlight_translation_id_6.html
Highlighted HTML saved to highlight_translation_id_7.html
Highlighted HTML saved to highlight_translation_id_19.html


**now show each html highlight**

In [74]:

for highlight in html_outputs_all_docs:
    print(f"translation_id: {highlight['translation_id']}")
    from IPython.display import HTML
    display(HTML(highlight['html_output']))

translation_id: 3


translation_id: 6


translation_id: 7


translation_id: 19


In [77]:
updated_spans_offsets_all_documents[3]

{'translation_id': 19,
 'updated_spans_offsets': [[{'text': ['vysílají'],
    'startnew': 4,
    'endnew': 12},
   {'text': ['vojáky,'], 'startnew': 31, 'endnew': 38},
   {'text': ['íránské'], 'startnew': 81, 'endnew': 88}],
  [{'text': ['odesílají'], 'startnew': 99, 'endnew': 108},
   {'text': ['jednotek', 'do', 'Saúdské', 'Arábie,'],
    'startnew': 184,
    'endnew': 211},
   {'text': ['odrazily'], 'startnew': 216, 'endnew': 224},
   {'text': ['agresivnější'], 'startnew': 231, 'endnew': 243},
   {'text': ['Íránu.'], 'startnew': 252, 'endnew': 258}],
  [{'text': ['další'], 'startnew': 295, 'endnew': 300},
   {'text': ['připraveny'], 'startnew': 331, 'endnew': 341},
   {'text': ['jít,'], 'startnew': 360, 'endnew': 364}],
  [{'text': ['"Jiné'], 'startnew': 380, 'endnew': 385},
   {'text': ['íránské', 'zneužívání'], 'startnew': 399, 'endnew': 417},
   {'text': ['regionu'], 'startnew': 426, 'endnew': 433},
   {'text': ['hledáme,'], 'startnew': 439, 'endnew': 447},
   {'text': ['přispěli'

**end**

In [60]:
from IPython.display import HTML
HTML(html_output)